In [ ]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy
from pydantic import BaseModel,Field


load_dotenv(override=True)

model = init_chat_model(
    model="deepseek:deepseek-v4-pro",
    extra_body={
        "thinking": {"type": "disabled"}  # 核心：关闭深度思考，ProviderStrategy才能生效
    }
)

def analysis(name:str) -> str:
    """分析顾客"""
    return f"李斯 , 普通用户 , 电话:19289883392 , 购买日期:2025-1-1 , 购买金额:300元"


class contentInfo(BaseModel):
    """用户信息"""
    name : str = Field(description="用户姓名")
    phone : str = Field(description="用户电话")
    data : str = Field(description="购买日期")
    amount : int = Field(description="购买金额")
    send : bool = Field(description="是否发邮件")


agent = create_agent(
    model= model,
    tools=[analysis],
    response_format=ToolStrategy(contentInfo)
)

from rich import print as rprint

res = agent.invoke({
    "messages" : [
        {"role":"user" , "content":"分析用户李斯(非vip用户不需要发邮件)"}
]
})

rprint(res)

